# Descriptive network pilot

This notebook reads only `data/relationships.csv`. It does not invent edges. Treat a small graph as illustrative unless the sample and coverage are adequate.


In [ ]:
from pathlib import Path
import pandas as pd
import networkx as nx

DATA_PATH = Path('../data/relationships.csv')
df = pd.read_csv(DATA_PATH)
required = {'project', 'institution_or_process', 'relationship_type', 'evidence_status'}
missing = required - set(df.columns)
if missing:
    raise ValueError(f'Missing required columns: {sorted(missing)}')
verified = df[df['evidence_status'].eq('verified')].copy()
if verified.empty:
    print('No verified relationships yet. Audit sources and add verified rows before analysing a network.')
else:
    print(f'Verified relationship rows: {len(verified)}')
    display(verified.head())


In [ ]:
if not verified.empty:
    G = nx.Graph()
    for _, row in verified.iterrows():
        project = f"project::{row['project']}"
        process = f"process::{row['institution_or_process']}"
        G.add_node(project, node_type='project', label=row['project'])
        G.add_node(process, node_type='process', label=row['institution_or_process'])
        G.add_edge(project, process, relationship_type=row['relationship_type'])
    print('Nodes:', G.number_of_nodes())
    print('Edges:', G.number_of_edges())
    print('Connected components:', nx.number_connected_components(G))
    print('Degree summary:', dict(G.degree()))
    # Do not report brokerage findings from a tiny pilot sample.
else:
    print('Analysis skipped: no verified rows.')


## Interpretation guardrails
- Missing public records are not proof of non-participation.
- Counts describe this coded sample only.
- Do not infer causation from network position.
- Do not report brokerage measures as findings from a handful of edges.
